# Net worth over time

Reads the latest run file under `data/shared/state/runs/` and plots month-end net worth (nominal) plus an inflation-adjusted (CPI) series.

The run file is opened read-only via `RunConnection`; all reads go through `rc.items(...)`, which returns tidy DataFrames suitable for pandas groupby / pivot. See [ADR-002](../docs/architecture/ADR-002-storage-schema.md) for the underlying schema.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analysis.sim.connection import RunConnection

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = REPO / "data" / "shared" / "state" / "runs"

run_files = sorted(RUNS_DIR.glob("*.sqlite3"))
if not run_files:
    raise FileNotFoundError(f"no run files under {RUNS_DIR}")
RUN_FILE = run_files[-1]
print(f"latest run: {RUN_FILE.name}")

rc = RunConnection(RUN_FILE)

In [ ]:
_META_COLS = [
    "scenario_name", "status", "started_at", "finished_at",
    "code_version", "code_dirty", "data_version", "data_dirty",
    "data_mode", "run_by",
]
run_meta = rc.items("simulation_run")[_META_COLS].copy()
run_meta["code_version"] = run_meta["code_version"].str[:12]
run_meta["data_version"] = run_meta["data_version"].str[:12]
run_meta.T


In [ ]:
nw = rc.items("net_worth").rename(columns={"net_worth": "nominal_cents"})
cpi = rc.items("net_worth_constant_cpi")[["month", "constant_cents"]]
nw = nw.merge(cpi, on="month", how="left")

nw["month"] = pd.to_datetime(nw["month"])
nw["nominal"] = nw["nominal_cents"] / 100
nw["cpi_constant"] = nw["constant_cents"].astype("Float64") / 100
nw = nw.set_index("month")
nw[["nominal", "cpi_constant"]].head()


## Year-end net worth (table)


In [ ]:
year_end = (
    nw.groupby(nw.index.year)
      .tail(1)
      .assign(year=lambda d: d.index.year)
      .set_index("year")[["nominal", "cpi_constant"]]
)
year_end.style.format({"nominal": "${:,.0f}", "cpi_constant": "${:,.0f}"})


## Monthly net worth (chart)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(nw.index, nw["nominal"], label="nominal")
if nw["cpi_constant"].notna().any():
    ax.plot(nw.index, nw["cpi_constant"], label="CPI-constant", linestyle="--")
ax.set_ylim(bottom=0)
ax.set_title(
    f"Net worth — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("month")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.xticks(rotation=90)
plt.show()

## Income by category

Income flows are transfers _out of_ accounts with `kind = 'income'`. Subkinds are bucketed:

- **salary** — `salary`
- **investment** — `interest`, `dividend`, `capital_gain`
- **other** — `ss_benefit`, `pension`, `rental`, `other_income`


In [ ]:
_INCOME_SUBKIND_TO_CAT = {
    "salary": "salary",
    "interest": "investment",
    "dividend": "investment",
    "capital_gain": "investment",
    "ss_benefit": "social security",
}
_INCOME_CATS = ["salary", "social security", "investment"]

xfer = rc.items("transfer_event", from_kind="income").copy()
xfer["month"] = pd.to_datetime(xfer["month"])
xfer["category"] = xfer["from_subkind"].map(_INCOME_SUBKIND_TO_CAT).fillna("other")

income = (
    xfer.groupby(["month", "category"])["amount"].sum()
        .unstack("category", fill_value=0)
        .reindex(columns=_INCOME_CATS, fill_value=0)
    / 100
)
income["total"] = income.sum(axis=1)
income.head()


In [ ]:
income_yearly = income.groupby(income.index.year).sum()
income_yearly.index.name = "year"

# Partial-first-year catch-up (salary/investment/SS already received before
# scenario.start) is authored as scenario data now, not back-calculated here:
# `ytd_amount`/`ytd_preroll` on scheduled.csv rows, or year_to_date.csv for
# everything else. The engine posts those as ordinary transfer_event rows
# (ADR-003 §4.11), so they're already included in `income` above.

income_yearly["total"] = income_yearly[_INCOME_CATS].sum(axis=1)
income_yearly.style.format("${:,.0f}")


## Income by type

(Not adjusted for inflation)

In [ ]:
categories = ["salary", "social security", "investment"]
fig, ax = plt.subplots(figsize=(10, 5))
income_yearly[categories].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#1f77b4", "#2ca02c", "#ff7f0e"],
)
ax.set_title(
    f"Annual income by category — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="category")
plt.show()


## Expenses by category

Expense flows are transfers _into_ accounts with `kind = 'expense'`. Subkinds are bucketed:

- **housing** — `housing`, `interest_expense`
- **healthcare** — `healthcare`
- **taxes** — `taxes_federal`, `taxes_state`, `taxes_property`
- **living** — `living`
- **other** — `other_expense`


In [ ]:
_EXPENSE_SUBKIND_TO_CAT = {
    "housing": "housing",
    "interest_expense": "housing",
    "taxes_property": "housing",
    "healthcare": "healthcare",
    "taxes_federal": "taxes",
    "taxes_state": "taxes",
    "living": "living",
    "food": "living",
}
_EXPENSE_CATS = ["housing", "healthcare", "taxes", "living", "other"]

xfer_exp = rc.items("transfer_event", to_kind="expense").copy()
xfer_exp["month"] = pd.to_datetime(xfer_exp["month"])
xfer_exp["category"] = (
    xfer_exp["to_subkind"].map(_EXPENSE_SUBKIND_TO_CAT).fillna("other")
)

expenses = (
    xfer_exp.groupby(["month", "category"])["amount"].sum()
        .unstack("category", fill_value=0)
        .reindex(columns=_EXPENSE_CATS, fill_value=0)
    / 100
)
expenses["total"] = expenses.sum(axis=1)
expenses_yearly = expenses.groupby(expenses.index.year).sum()
expenses_yearly.index.name = "year"
expenses_yearly.style.format("${:,.0f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
expenses_yearly[_EXPENSE_CATS].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#1f77b4", "#d62728", "#9467bd", "#8c564b", "#7f7f7f"],
)
ax.set_title(
    f"Annual expenses by category — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="category")
plt.show()


## Realized vs. unrealized gains

Splits every income-side flow into three buckets, to check whether actual cash distributions (not paper growth) cover expenses:

- **taxable** — reaches a liquid account (`checking`/`savings`) with a taxable character (`ordinary`, `ltcg`, `ss_benefit`, `tax_deferred_out`).
- **non-taxable** — reaches a liquid account with a `tax_free` character (Roth growth/withdrawals, tax-free income).
- **unrealized** — income-kind growth (interest, dividends, appreciation) that stays inside a retirement/brokerage account and never reaches cash.

Caveats (this is a first pass — flag anything that doesn't match your mental model):

- `return_of_capital` (basis returned on a sale) and `tax_deferred_in` (contributions) are excluded entirely: they're principal movements, not gains.
- "Liquid" = `checking`/`savings` subkinds only; a distribution landing in, say, an HSA cash account wouldn't count as realized here.
- `distribution_coverage` below is `taxable + non-taxable − expenses`; negative years are ones where realized cash distributions alone didn't cover spending (the gap was plugged by `waterfall_fill`/`refill_transfer` principal draws, which aren't gains and so aren't counted as coverage here).


In [ ]:
LIQUID_SUBKINDS = {"checking", "savings"}
TAXABLE_GAIN_CHARS = {"ordinary", "ltcg", "ss_benefit", "tax_deferred_out"}
NONTAXABLE_GAIN_CHARS = {"tax_free"}

gains = rc.items("transfer_event")
gains = gains[gains["reason_kind"] != "initial"].copy()
gains["month"] = pd.to_datetime(gains["month"])

reaches_cash = gains["to_subkind"].isin(LIQUID_SUBKINDS)
is_unrealized = (gains["from_kind"] == "income") & ~reaches_cash
is_taxable = reaches_cash & gains["tax_character"].isin(TAXABLE_GAIN_CHARS)
is_nontaxable = reaches_cash & gains["tax_character"].isin(NONTAXABLE_GAIN_CHARS)

gains["gain_category"] = pd.NA
gains.loc[is_unrealized, "gain_category"] = "unrealized"
gains.loc[is_taxable, "gain_category"] = "taxable"
gains.loc[is_nontaxable, "gain_category"] = "non-taxable"
gains = gains.dropna(subset=["gain_category"])

_GAIN_CATS = ["taxable", "non-taxable", "unrealized"]
gains_yearly = (
    gains.groupby([gains["month"].dt.year, "gain_category"])["amount"].sum()
        .unstack("gain_category", fill_value=0)
        .reindex(columns=_GAIN_CATS, fill_value=0)
    / 100
)
gains_yearly.index.name = "year"
gains_yearly["realized_total"] = gains_yearly["taxable"] + gains_yearly["non-taxable"]
gains_yearly["expenses"] = expenses_yearly["total"].reindex(gains_yearly.index).fillna(0)
gains_yearly["distribution_coverage"] = gains_yearly["realized_total"] - gains_yearly["expenses"]
gains_yearly.style.format("${:,.0f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
years = gains_yearly.index.astype(str)
gains_yearly[_GAIN_CATS].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#2ca02c", "#1f77b4", "#7f7f7f"],
)
ax.plot(
    years, gains_yearly["expenses"],
    label="expenses", color="#d62728", marker="o", linewidth=2,
)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title(
    f"Realized vs. unrealized gains — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="category")
plt.setp(ax.get_xticklabels(), rotation=90)
plt.show()


## After-tax income

After-tax income is gross income less the year's computed federal and state tax liabilities. `tax_free` priors are excluded: they are tax inputs, not a substitute for after-tax income.

> **Tax timing:** tax liability is attributed to the tax year in which income occurs. Withholding and estimated payments are cash movements in that same year; the annual reconciliation payment or refund occurs in April of the following year. Do not interpret the reconciliation year as the tax year.

Tax rules reconcile the previous calendar year in April. The scenario's final year therefore has no computed tax liability and is left blank rather than shown as zero.

The final column compares after-tax income with non-tax expenses, so estimated tax liabilities are not counted twice.

In [ ]:
tax_summary = rc.items("tax_year_summary")
tax_liability = (
    tax_summary.loc[
        tax_summary["bucket"].isin(["federal_tax_liability", "state_tax_liability"]),
        ["year", "amount"],
    ]
    .groupby("year")["amount"]
    .sum()
    / 100
)

tax_reconciliation = (
    tax_summary.loc[
        tax_summary["bucket"].isin([
            "federal_tax_liability", "state_tax_liability",
            "federal_tax_paid", "state_tax_paid",
            "federal_tax_due", "state_tax_due",
            "federal_tax_overpaid", "state_tax_overpaid",
        ]),
        ["year", "bucket", "amount"],
    ]
    .pivot_table(index="year", columns="bucket", values="amount", aggfunc="sum", fill_value=0)
    .rename(columns={
        "federal_tax_liability": "federal_liability",
        "state_tax_liability": "state_liability",
        "federal_tax_paid": "federal_withheld_or_paid",
        "state_tax_paid": "state_withheld_or_paid",
        "federal_tax_due": "federal_reconciliation_due",
        "state_tax_due": "state_reconciliation_due",
        "federal_tax_overpaid": "federal_estimated_refund",
        "state_tax_overpaid": "state_estimated_refund",
    })
    / 100
)
tax_reconciliation["reconciliation_year"] = tax_reconciliation.index + 1
money_columns = [
    column for column in tax_reconciliation.columns
    if column != "reconciliation_year"
]
display(tax_reconciliation.style.format({column: "${:,.0f}" for column in money_columns}))

net = pd.DataFrame({
    "gross_income": income_yearly["total"],
    "tax_liability": tax_liability,
    "expenses": expenses_yearly["total"],
    "tax_expenses": expenses_yearly["taxes"],
})
net[["gross_income", "expenses", "tax_expenses"]] = net[[
    "gross_income", "expenses", "tax_expenses",
]].fillna(0)
net["after_tax_income"] = net["gross_income"] - net["tax_liability"]
net["non_tax_expenses"] = net["expenses"] - net["tax_expenses"]
net["after_tax_surplus"] = net["after_tax_income"] - net["non_tax_expenses"]
net.style.format("${:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
years = net.index.astype(str)
ax.bar(
    years, net["after_tax_income"],
    label="after-tax income", color="#2ca02c", alpha=0.7,
 )
ax.bar(
    years, -net["non_tax_expenses"],
    label="non-tax expenses", color="#d62728", alpha=0.7,
 )
ax.plot(
    years, net["tax_liability"],
    label="tax liability", color="#9467bd", marker="o", linewidth=2,
 )
ax.plot(
    years, net["after_tax_surplus"],
    label="after-tax surplus", color="black", marker="o", linewidth=2,
 )
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title(
    f"Annual after-tax income — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("year")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=90)
plt.show()